# IGNIS Fire Calendar — exploratory analysis

Walk through the 26-year harmonized MODIS + VIIRS burning-activity record:
seasonality, anomalies, trends, sensor harmonization and a mini segmentation of a live-day hotspot set.

**Data:** `data/calendar_<region>.json` in [Nabidnur/ignis-fire-calendar](https://huggingface.co/datasets/Nabidnur/ignis-fire-calendar) · **App:** [ignis-spaceapps2026.vercel.app](https://ignis-spaceapps2026.vercel.app)

In [ ]:
import json, math, urllib.request
import matplotlib.pyplot as plt

def load(name):
    url = f"https://huggingface.co/datasets/Nabidnur/ignis-fire-calendar/resolve/main/data/{name}"
    return json.load(urllib.request.urlopen(url))

cal = load("calendar_bangladesh.json")
outlook = load("outlook_bangladesh.json")
print(cal['name'], '| sensors:', list(cal['sensors'].keys()))
print('harmonization:', {k: cal['harmonization'][k] for k in ('overlapMonths', 'meanRatioSNPPtoMODIS')})

## 1. Unified 26-year series + climatology z-anomalies

In [ ]:
import numpy as np

rows = {}
for sensor, s in cal['sensors'].items():
    for m in s['monthly']:
        rows.setdefault(m['ym'], 0.0)
        rows[m['ym']] += m['est']          # harmonized: VIIRS-era adds VIIRS, MODIS-era is MODIS (scaled)
yms = sorted(rows)
y = np.array([rows[k] for k in yms])

month = np.array([int(k[5:7]) for k in yms])
clim = np.array([np.nanmean(y[month == m]) for m in range(1, 13)])
std  = np.array([np.nanstd(y[month == m])  for m in range(1, 13)])
z = (y - clim[month - 1]) / (std[month - 1] + 1e-9)

fig, ax = plt.subplots(figsize=(13, 4), constrained_layout=True)
ax.plot(range(len(y)), y, lw=.9, color='#94A3B8', label='monthly detections (harmonized)')
hot = np.abs(z) >= 2
ax.scatter(np.where(hot)[0], y[hot], color='#EF4444', s=22, zorder=3, label='unusual month (|z|≥2)')
step = max(1, len(yms)//10)
ax.set_xticks(range(0, len(yms), step)); ax.set_xticklabels([yms[i][:4] for i in range(0, len(yms), step)])
ax.set_title(f"{cal['name']} — 26-year harmonized fire activity"); ax.legend()
plt.show()
print(f"peak climatological month: {['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][int(np.nanargmax(clim))]}  ({clim.max():.0f} avg detections)")

## 2. Seasonal fingerprint (why harmonization matters in Bangladesh)
Boro rice-residue burning peaks Mar–Apr; Aman residue Nov; monsoon lull Jun–Sep. The NASA record reproduces the agronomy.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2), constrained_layout=True)
ax.bar(range(1, 13), clim, color=['#F97316' if c > clim.max()*.75 else '#FDBA74' for c in clim])
ax.set_xticks(range(1, 13)); ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax.set_title('Climatological monthly detections — Bangladesh'); plt.show()

## 3. Outlook (harmonic regression, shipped in `outlook_*.json`)

In [ ]:
hist = outlook['history']; fc = outlook['forecast']
fig, ax = plt.subplots(figsize=(13, 3.6), constrained_layout=True)
ax.plot([h['ym'] for h in hist[-60:]], [h['est'] for h in hist[-60:]], color='#38BDF8', label='history')
f_ym = [f['ym'] for f in fc]; f_mu = [f['mean'] for f in fc]
ax.plot(f_ym, f_mu, color='#F97316', label='forecast')
ax.fill_between(f_ym, [f['lo'] for f in fc], [f['hi'] for f in fc], color='#F97316', alpha=.2, label='uncertainty')
ax.legend(); ax.set_title(f"{outlook['region']} — 12-month outlook (peak: {', '.join(outlook['peakMonths'])})")
plt.xticks(rotation=45, fontsize=7); plt.show()

## 4. Mini fire-segmentation demo (the algorithm IGNIS runs in-browser)
Grid-accelerated DBSCAN + convex hull — the same logic as `src/lib/ignis/segmentation.ts`.

In [ ]:
# Synthetic demonstration around a persistent cluster (live fetching needs FIRMS/GIBS network access —
# see the app for the live version). Replace with a FIRMS CSV download for a real-day replay.
rng = np.random.default_rng(7)
pts = np.vstack([rng.normal([90.3, 23.7], .12, (160, 2)), rng.normal([89.2, 22.4], .3, (60, 2)), rng.uniform([88.5, 21.0], [92.5, 26.0], (40, 2))])

def dbscan(P, eps, min_pts):
    n = len(P); lab = np.full(n, -2)
    cid = 0
    for i in range(n):
        if lab[i] != -2: continue
        nb = [j for j in range(n) if j != i and ((P[j]-P[i])**2).sum() <= eps*eps]
        if len(nb)+1 < min_pts: lab[i] = -1; continue
        lab[i] = cid; q = list(nb)
        k = 0
        while k < len(q):
            j = q[k]; k += 1
            if lab[j] == -1: lab[j] = cid
            if lab[j] != -2: continue
            lab[j] = cid
            nb2 = [x for x in range(n) if x != j and ((P[x]-P[j])**2).sum() <= eps*eps]
            if len(nb2)+1 >= min_pts: q += nb2
        cid += 1
    return lab, cid

lab, k = dbscan(pts, eps=.35, min_pts=4)
fig, ax = plt.subplots(figsize=(6.5, 5.5), constrained_layout=True)
noise = lab == -1
ax.scatter(pts[noise,0], pts[noise,1], s=8, color='#94A3B8', label='scattered')
for c in range(k):
    m = lab == c
    ax.scatter(pts[m,0], pts[m,1], s=12, alpha=.8)
    if m.sum() >= 8:
        cx, cy = pts[m,0].mean(), pts[m,1].mean()
        ax.annotate(f'cluster {c} · {m.sum()} det', (cx, cy), color='white', ha='center',
                    bbox=dict(boxstyle='round', fc='#DC2626', alpha=.85, pad=.3))
ax.set_title('DBSCAN fire-cluster segmentation + boundary identification'); ax.legend(); plt.show()

## Citation
If you use these calendars: `IGNIS Fire Calendar (2026), NASA Space Apps Challenge 2026, https://huggingface.co/datasets/Nabidnur/ignis-fire-calendar`